# Tahap 2e — ICC dari 200 teks retest
**CPU biasa, <2 menit.** Menghitung ICC antar-sesi (anotasi asli vs anotasi ulang oleh psikolog yang sama) per kategori, dengan CI 95% bootstrap. Kappa yang sudah ada tidak diubah.
Notebook mencari file `retest_200.csv` di Drive dan mendeteksi kolom skor secara otomatis. Jika deteksi gagal, ia mencetak nama kolom dan Anda mengisi `MAP` secara manual.

In [ ]:
import os, json, re, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'
DATA4=f'{DRIVE}/JURNAL-DATASET'
OUT4=f'{DRIVE}/JURNAL-OUTPUTS/laptop_jurnal'
PREP4=f'{OUT4}/prep'; PRED4=f'{OUT4}/preds'; S2=f'{OUT4}/stage2'
os.makedirs(S2,exist_ok=True)
def P4(fn):
    p=f'{DATA4}/{fn}'
    return p if os.path.exists(p) else f'{DRIVE}/tesis-dataset 3/{fn}'
CATEGORIES=['Intrusive','Avoidance','Arousal','NegCogMood']
SEED=42; random.seed(SEED); np.random.seed(SEED)
TAG='jurnal_final'
print('setup OK')

In [ ]:
import numpy as np, pandas as pd
def icc_all(X):
    """X: n x 2 (sesi 1, sesi 2). Mengembalikan ICC(2,1) absolute agreement, ICC(3,1) consistency, dan versi rata-rata (k=2)."""
    X=np.asarray(X,float); n,k=X.shape; gm=X.mean(); rm=X.mean(1); cm=X.mean(0)
    msr=k*((rm-gm)**2).sum()/(n-1); msc=n*((cm-gm)**2).sum()/(k-1)
    mse=((X-rm[:,None]-cm[None,:]+gm)**2).sum()/((n-1)*(k-1))
    return {'ICC2_1_abs_agreement':(msr-mse)/(msr+(k-1)*mse+k*(msc-mse)/n),
            'ICC3_1_consistency':(msr-mse)/(msr+(k-1)*mse),
            'ICC2_k_abs_agreement':(msr-mse)/(msr+(msc-mse)/n),
            'ICC3_k_consistency':(msr-mse)/msr}
def icc_ci(X,B=2000,seed=42):
    rng=np.random.default_rng(seed); X=np.asarray(X,float); n=len(X); base=icc_all(X); bs={k:[] for k in base}
    for _ in range(B):
        r=icc_all(X[rng.integers(0,n,n)])
        for k,v in r.items(): bs[k].append(v)
    return {k:(base[k],np.nanpercentile(bs[k],2.5),np.nanpercentile(bs[k],97.5)) for k in base}

In [ ]:
import glob
cands=[]
for base in (DATA4,OUT4,DRIVE):
    cands+=glob.glob(f'{base}/**/*retest*200*.csv',recursive=True)+glob.glob(f'{base}/**/retest*.csv',recursive=True)
cands=sorted(set(cands)); print('kandidat file:',cands)
RETEST_PATH=cands[0] if cands else None   # ganti manual jika perlu, mis. RETEST_PATH='/content/drive/MyDrive/....csv'
df=pd.read_csv(RETEST_PATH,encoding='utf-8-sig'); print(RETEST_PATH, df.shape); print(list(df.columns))

In [ ]:
# MAP manual (opsional): {'Intrusive':('kolom_asli','kolom_ulang'), ...}. Biarkan None untuk deteksi otomatis.
MAP = {c: (f'orig_{c}', f'retest_{c}') for c in CATEGORIES}
RE_KEYS=('retest','ulang','re_','_re','_2','2nd','baru','new','sesi2','s2')
def autodetect(df):
    m={}
    for c in CATEGORIES:
        cols=[x for x in df.columns if c.lower() in x.lower() and pd.api.types.is_numeric_dtype(df[x])]
        if len(cols)!=2: return None,f'{c}: ditemukan {cols}'
        second=[x for x in cols if any(k in x.lower() for k in RE_KEYS)]
        first=[x for x in cols if x not in second]
        if len(second)!=1 or len(first)!=1: return None,f'{c}: tidak bisa membedakan sesi dari {cols}'
        m[c]=(first[0],second[0])
    return m,None
if MAP is None:
    MAP,err=autodetect(df)
    if MAP is None: raise SystemExit('Deteksi otomatis gagal -> '+err+'\nIsi MAP manual di sel ini lalu jalankan ulang.')
print(MAP)

In [ ]:
rows=[]
for c,(a,b) in MAP.items():
    X=df[[a,b]].dropna().values; res=icc_ci(X)
    row={'Category':c,'n':len(X),'Spearman':pd.Series(X[:,0]).corr(pd.Series(X[:,1]),method='spearman'),'MeanAbsDiff':np.abs(X[:,0]-X[:,1]).mean(),
         'Binary_agreement(>=0.5)':((X[:,0]>=0.5)==(X[:,1]>=0.5)).mean()}
    for k,(v,lo,hi) in res.items(): row[k]=v; row[k+'_CI']=f'[{lo:.2f}, {hi:.2f}]'
    rows.append(row)
icc=pd.DataFrame(rows); print(icc.round(3).T.to_string()); icc.to_csv(f'{S2}/stage2e_icc_retest.csv',index=False)
print('\nPedoman umum (Koo & Li, 2016): <0.5 buruk, 0.5-0.75 sedang, 0.75-0.9 baik, >0.9 sangat baik.')

Selesai. Kirim screenshot tabel hasil (atau file `stage2e_icc_retest.csv`). Tidak ada teks asli yang perlu dikirim.